# 7 - Agentes no LangChain

Um **agente** é um LLM que raciocina em loop: ele decide qual ação tomar, executa uma tool,
observa o resultado, e repete até ter informação suficiente para responder.

É diferente de uma chain simples onde o fluxo é fixo (`A → B → C`).
No agente, o **LLM decide o fluxo** em tempo de execução:

```
Pergunta
   │
   ▼
 [ LLM ]  ──► "preciso buscar o usuário"  ──►  buscar_usuario("Maria")
   │                                                    │
   │◄───────────────── "Maria tem 30 anos" ─────────────┘
   │
   ▼
 [ LLM ]  ──► "agora preciso somar"  ──►  calcular_soma([5, 10, 15])
   │                                              │
   │◄──────────────── "A soma é 30" ──────────────┘
   │
   ▼
 Resposta final: "Maria tem 30 anos e a soma é 30"
```

### O que mudou na API moderna

| Antigo (deprecado) | Moderno |
|--------------------|--------|
| `from langchain.llms import OpenAI` | `from langchain_openai import ChatOpenAI` |
| `from langchain.agents import initialize_agent` | `from langgraph.prebuilt import create_react_agent` |
| `from langchain.agents import AgentType` | não necessário |
| `Tool(name=..., func=..., description=...)` | `@tool` decorator (notebook 4) |
| `agent.run(query)` | `agent.invoke({"messages": query})` |

> **LangGraph**: o LangChain moderno delega a lógica de agentes ao **LangGraph**,
> uma biblioteca separada do mesmo ecossistema. O `create_react_agent` do LangGraph
> substitui o `initialize_agent` com suporte a streaming, memória e muito mais.

### Pacotes necessários

```bash
pip install langchain-openai langgraph pydantic python-dotenv
```

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# ChatOpenAI substitui o antigo OpenAI(temperature=0)
# ChatOpenAI usa os modelos de chat (gpt-4o-mini, etc.)
# O antigo OpenAI usava modelos de completion (text-davinci), que foram descontinuados
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. Agente simples com uma tool

Vamos começar com o exemplo mais básico: um agente com uma única ferramenta.

O padrão ReAct (**Re**asoning + **Act**ing) é o mais comum:
o modelo alterna entre **pensar** (Thought) e **agir** (Action/Observation)
até chegar a uma resposta final.

> **`create_react_agent`**: cria um agente ReAct pronto para uso.
> Substitui o antigo `initialize_agent(..., agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION)`.

In [2]:
from langchain_core.tools import tool
from langchain.agents import create_agent

# Definindo a tool com o decorator moderno
# Substitui o antigo: Tool(name=..., func=resposta_simples, description=...)
@tool
def resposta_simples(query: str) -> str:
    """Ferramenta simples para responder perguntas diretas sobre capitais de países."""
    return f"A resposta para '{query}' é Paris."

# Criando o agente
# Substitui: initialize_agent(tools, llm, agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION)
agent = create_agent(llm, tools=[resposta_simples])

print("Agente criado com tools:", [resposta_simples.name])

Agente criado com tools: ['resposta_simples']


In [3]:
# Executando o agente
# Substitui o antigo: agent.run(input_query)
# Agora usamos .invoke() com um dicionário de mensagens
resposta = agent.invoke({"messages": "Qual é a capital da França?"})

# A resposta final está na última mensagem do histórico
print(resposta["messages"][-1].content)

A capital da França é Paris.


## 2. Inspecionando o raciocínio do agente

Um dos pontos fortes do LangGraph é a visibilidade sobre cada passo do agente.
Podemos ver todas as mensagens trocadas — incluindo as `ToolMessage` com os resultados
das tools — que formam o histórico completo do raciocínio.

| Tipo de mensagem | Quem enviou | Conteúdo |
|------------------|-------------|----------|
| `HumanMessage` | usuário | a pergunta original |
| `AIMessage` | LLM | raciocínio + tool_calls |
| `ToolMessage` | executor | resultado da tool |
| `AIMessage` (final) | LLM | resposta final ao usuário |

In [4]:
# Inspecionando todas as mensagens do ciclo
for msg in resposta["messages"]:
    tipo = type(msg).__name__
    conteudo = msg.content or str(getattr(msg, 'tool_calls', ''))
    print(f"[{tipo}]")
    print(f"  {conteudo[:200]}")
    print()

[HumanMessage]
  Qual é a capital da França?

[AIMessage]
  [{'name': 'resposta_simples', 'args': {'query': 'capital da França'}, 'id': 'call_CIO7uGPTfv89BMlJGChoaCU4', 'type': 'tool_call'}]

[ToolMessage]
  A resposta para 'capital da França' é Paris.

[AIMessage]
  A capital da França é Paris.



## 3. Agente com múltiplas tools e banco de dados

Agentes ficam mais poderosos com múltiplas tools.
O LLM decide **qual** usar e **em que ordem**, podendo combinar resultados de ferramentas diferentes.

Neste exemplo criamos duas tools:
- `buscar_usuario`: consulta um banco SQLite em memória
- `calcular_soma`: soma uma lista de números

In [5]:
import sqlite3
from pydantic import BaseModel, Field
from typing import List

# --- Banco de dados em memória ---
def create_database():
    conn = sqlite3.connect(":memory:")
    c = conn.cursor()
    c.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT, age INTEGER)")
    c.execute("INSERT INTO users (name, age) VALUES ('João', 25), ('Maria', 30)")
    conn.commit()
    return conn

# --- Schemas Pydantic para os argumentos ---
class BuscarUsuarioArgs(BaseModel):
    user_name: str = Field(description="Nome do usuário a buscar no banco de dados")

class CalcularSomaArgs(BaseModel):
    numbers: List[float] = Field(description="Lista de números a serem somados")

# --- Definição das tools ---
@tool(args_schema=BuscarUsuarioArgs)
def buscar_usuario(user_name: str) -> str:
    """Procura informações sobre um usuário na base de dados pelo nome."""
    conn = create_database()
    c = conn.cursor()
    c.execute("SELECT name, age FROM users WHERE name = ?", (user_name,))
    result = c.fetchone()
    conn.close()
    if result:
        return f"{result[0]} tem {result[1]} anos"
    return "Usuário não encontrado"

@tool(args_schema=CalcularSomaArgs)
def calcular_soma(numbers: List[float]) -> str:
    """Calcula a soma de uma lista de números."""
    total = sum(numbers)
    return f"A soma dos números {numbers} é {total}"

print("Tools criadas:", [buscar_usuario.name, calcular_soma.name])

Tools criadas: ['buscar_usuario', 'calcular_soma']


## 4. Executando o agente com múltiplas tools

Quando a pergunta exige **mais de uma tool**, o agente executa um ciclo para cada uma.
Cada iteração: LLM raciocina → chama tool → recebe resultado → raciocina novamente.

> **Correção importante**: o código original tinha um bug na `sum_numbers` que usava `eval()`
> para converter a string — frágil e inseguro. Com `args_schema=CalcularSomaArgs` e
> `List[float]`, o Pydantic faz a conversão de forma segura e o LLM já passa uma lista real.

In [6]:
# Criando o agente com as duas tools
agent_multi = create_agent(llm, tools=[buscar_usuario, calcular_soma])

# Pergunta que exige duas tools diferentes
input_query = "Qual é a soma de 5, 10 e 15? Quantos anos Maria tem?"

resposta_multi = agent_multi.invoke({"messages": input_query})

# Resposta final
print(resposta_multi["messages"][-1].content)

A soma de 5, 10 e 15 é 30. Maria tem 30 anos.


In [7]:
# Inspecionando o ciclo completo de raciocínio
print(f"Total de mensagens no ciclo: {len(resposta_multi['messages'])}\n")

for i, msg in enumerate(resposta_multi["messages"]):
    tipo = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        # Mensagem do LLM chamando uma tool
        for tc in msg.tool_calls:
            print(f"[{i}] {tipo} → chama '{tc['name']}' com args: {tc['args']}")
    elif hasattr(msg, 'name') and msg.name:  # ToolMessage
        print(f"[{i}] {tipo} ({msg.name}) → '{msg.content}'")
    else:
        conteudo = msg.content[:100] if msg.content else '(vazio)'
        print(f"[{i}] {tipo} → '{conteudo}'") 

Total de mensagens no ciclo: 5

[0] HumanMessage → 'Qual é a soma de 5, 10 e 15? Quantos anos Maria tem?'
[1] AIMessage → chama 'calcular_soma' com args: {'numbers': [5, 10, 15]}
[1] AIMessage → chama 'buscar_usuario' com args: {'user_name': 'Maria'}
[2] ToolMessage (calcular_soma) → 'A soma dos números [5.0, 10.0, 15.0] é 30.0'
[3] ToolMessage (buscar_usuario) → 'Maria tem 30 anos'
[4] AIMessage → 'A soma de 5, 10 e 15 é 30. Maria tem 30 anos.'


## 5. Streaming do raciocínio em tempo real

Uma vantagem do LangGraph é o suporte nativo a **streaming**.
Com `.stream()`, você recebe cada passo do agente conforme ele acontece,
em vez de esperar o ciclo completo terminar.

Útil para interfaces de chat onde você quer mostrar o "pensamento" do agente ao vivo.

In [8]:
# .stream() emite cada passo do agente conforme acontece
print(f"Pergunta: '{input_query}'\n")
print("=" * 50)

for step in agent_multi.stream({"messages": input_query}):
    # Cada step é um dict com o nó do grafo que gerou o evento
    for node, data in step.items():
        print(f"\n[Nó: {node}]")
        for msg in data.get("messages", []):
            tipo = type(msg).__name__
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  → Chamando '{tc['name']}' com {tc['args']}")
            elif msg.content:
                print(f"  [{tipo}] {msg.content[:150]}")

Pergunta: 'Qual é a soma de 5, 10 e 15? Quantos anos Maria tem?'


[Nó: model]
  → Chamando 'calcular_soma' com {'numbers': [5, 10, 15]}
  → Chamando 'buscar_usuario' com {'user_name': 'Maria'}

[Nó: tools]
  [ToolMessage] A soma dos números [5.0, 10.0, 15.0] é 30.0

[Nó: tools]
  [ToolMessage] Maria tem 30 anos

[Nó: model]
  [AIMessage] A soma de 5, 10 e 15 é 30. Maria tem 30 anos.


## Resumo

| Conceito | Descrição |
|----------|-----------|
| `create_react_agent(llm, tools)` | Cria um agente ReAct — substitui `initialize_agent` |
| `agent.invoke({"messages": query})` | Executa o agente — substitui `agent.run()` |
| `agent.stream({"messages": query})` | Executa com streaming passo a passo |
| `HumanMessage` | Pergunta do usuário |
| `AIMessage` com `tool_calls` | LLM decidiu chamar uma tool |
| `ToolMessage` | Resultado da execução da tool devolvido ao LLM |
| `AIMessage` final | Resposta final ao usuário após todos os ciclos |

**Ciclo ReAct completo:**

```python
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def minha_tool(query: str) -> str:
    """Descrição de quando usar esta tool."""
    return "resultado"

llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_react_agent(llm, tools=[minha_tool])

# Execução completa
resposta = agent.invoke({"messages": "sua pergunta"})
print(resposta["messages"][-1].content)

# Execução com streaming
for step in agent.stream({"messages": "sua pergunta"}):
    print(step)
```

> **Próximo passo**: adicionar **memória** ao agente para que ele lembre de conversas anteriores,
> usando o `checkpointer` do LangGraph.